=============================================================================
STUDENT 3 — IMAGE ANALYST (BUILT FROM SCRATCH)
Multimodal Crime / Incident Report Analyzer
AI for Engineers — Group Assignment
=============================================================================
DATASET: Roboflow Fire Detection Dataset
Uses Roboflow's trained YOLOv8 model for fire and smoke detection
HOW TO USE:
1. Upload this notebook to Google Colab
2. Make sure your images are in Google Drive
3. Get your Roboflow API key from app.roboflow.com
4. Paste your API key in CELL 4
5. Run All cells

STRUCTURE (following assignment tip):
PART 1 → Object Detection using Roboflow YOLOv8 fire model
PART 2 → Scene Classification + OCR on top of detections
=============================================================================

### Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Install All Dependencies

In [2]:
import subprocess
subprocess.run(["pip", "install", "roboflow", "ultralytics",
                "opencv-python-headless", "pytesseract",
                "pandas", "numpy", "pillow", "-q"])
subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "-q"])
print("✔ All dependencies installed")

✔ All dependencies installed


### Imports

In [3]:
import os
import cv2
import glob
import warnings
import numpy as np
import pandas as pd
import pytesseract
from PIL import Image
from roboflow import Roboflow
from IPython.display import display

warnings.filterwarnings("ignore")
print("✔ Imports done")

✔ Imports done


### Configuration — SET YOUR API KEY AND PATH HERE

In [4]:
# Get your API key from app.roboflow.com → profile → API Keys
ROBOFLOW_API_KEY  = "yhOixp9dJyKulw396HZ0"       # ← paste your key here

IMAGE_DATA_DIR    = "/content/drive/MyDrive/AI_DATASETS/IMAGE_AI"
OUTPUT_CSV        = "/content/drive/MyDrive/AI_DATASETS/IMAGE_AI.csv"

CONFIDENCE_THRESH = 40       # Roboflow uses 0-100 scale (40 = 40%)
OVERLAP_THRESH    = 30       # Overlap threshold for bounding boxes
IMAGE_EXTENSIONS  = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]

### Preview Image Files

In [5]:
print("=" * 55)
print("  YOUR IMAGE FILES")
print("=" * 55)

image_files = []
for ext in IMAGE_EXTENSIONS:
    image_files += glob.glob(os.path.join(IMAGE_DATA_DIR, ext))
    image_files += glob.glob(os.path.join(IMAGE_DATA_DIR, "**", ext), recursive=True)
image_files = sorted(set(image_files))

if image_files:
    print(f"  Total images found: {len(image_files)}")
    print("\n  Sample files:")
    for f in image_files[:5]:
        print(f"    🖼 {os.path.basename(f)}")
else:
    print("  ⚠ No images found — check your path in CELL 4")

# =============================================================================
#  PART 1 — OBJECT DETECTION using Roboflow YOLOv8 Fire Model
#  Following assignment tip: use Roboflow starter model
# =============================================================================

  YOUR IMAGE FILES
  Total images found: 1079

  Sample files:
    🖼 fire-1-_preprocessed_jpg.rf.1c31de478d640135d96dc86fa92fc14a.jpg
    🖼 fire-10-_preprocessed_jpg.rf.86825aec42445b822802d8fa920de76e.jpg
    🖼 fire-100-_preprocessed_jpg.rf.2ca6985df2bb59555acfed7aad3b08b0.jpg
    🖼 fire-103-_preprocessed_jpg.rf.57d3812d109aa3325a6bd37c6deda059.jpg
    🖼 fire-104-_preprocessed_jpg.rf.56c40df80a1612af40abeb802403699c.jpg


### Load Roboflow YOLOv8 Fire Detection Model

In [6]:
print("\n" + "=" * 55)
print("  PART 1 — OBJECT DETECTION (Roboflow YOLOv8)")
print("=" * 55)

print("Loading Roboflow fire detection model...")
rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("rahuls-workspace-4h5k3").project("fire-detection-rsqrr-uewfo")
version = project.version(1)
model   = version.model
print("✔ Roboflow YOLOv8 fire model loaded")
print("  Classes: fire, smoke\n")


  PART 1 — OBJECT DETECTION (Roboflow YOLOv8)
Loading Roboflow fire detection model...
loading Roboflow workspace...
loading Roboflow project...
✔ Roboflow YOLOv8 fire model loaded
  Classes: fire, smoke



### Run Detection on All Images

In [7]:
def run_roboflow_detection(image_path):
    """
    Run Roboflow YOLOv8 fire detection model on a single image.
    Returns detected labels, bounding box summary, and confidence.
    """
    result      = model.predict(image_path,
                                confidence=CONFIDENCE_THRESH,
                                overlap=OVERLAP_THRESH).json()
    predictions = result.get("predictions", [])

    labels = []
    confs  = []
    boxes  = []

    for pred in predictions:
        label = pred["class"]
        conf  = round(pred["confidence"], 2)
        x, y  = pred["x"], pred["y"]
        w, h  = pred["width"], pred["height"]

        labels.append(label)
        confs.append(conf)
        boxes.append({"label": label, "conf": conf,
                       "x": x, "y": y, "w": w, "h": h})

    avg_conf = round(float(np.mean(confs)), 2) if confs else 0.0
    return labels, boxes, avg_conf


def format_bounding_boxes(boxes):
    """
    Format bounding boxes as human readable string.
    Example: '2 fire regions, 1 smoke plume'
    """
    if not boxes:
        return "None"

    label_counts = {}
    for b in boxes:
        label_counts[b["label"]] = label_counts.get(b["label"], 0) + 1

    label_names = {
        "fire":  "fire region",
        "smoke": "smoke plume",
    }

    parts = []
    for label, count in label_counts.items():
        name = label_names.get(label, label)
        parts.append(f"{count} {name}{'s' if count > 1 else ''}")

    return ", ".join(parts)


# Run detection on all images
print(f"Running Roboflow YOLOv8 detection on {len(image_files)} images...\n")

detection_results = []

if not image_files:
    print("⚠ No images found — demo data will be used")
else:
    for i, img_path in enumerate(image_files):
        img_id = f"IMG_{i+1:03d}"
        try:
            labels, boxes, avg_conf = run_roboflow_detection(img_path)
            detection_results.append({
                "Image_ID":   img_id,
                "image_path": img_path,
                "labels":     labels,
                "boxes":      boxes,
                "avg_conf":   avg_conf,
            })
        except Exception as e:
            print(f"  ❌ {img_id} failed: {e}")

        if (i + 1) % 50 == 0:
            print(f"  ✔ {i+1}/{len(image_files)} images processed...")

    print(f"\n✔ PART 1 COMPLETE — {len(detection_results)} images detected")

# =============================================================================
#  PART 2 — SCENE CLASSIFICATION + OCR
#  On top of Roboflow YOLOv8 detection results
# =============================================================================

Running Roboflow YOLOv8 detection on 1079 images...

  ✔ 50/1079 images processed...
  ✔ 100/1079 images processed...
  ✔ 150/1079 images processed...
  ✔ 200/1079 images processed...
  ✔ 250/1079 images processed...
  ✔ 300/1079 images processed...
  ✔ 350/1079 images processed...
  ✔ 400/1079 images processed...
  ✔ 450/1079 images processed...
  ✔ 500/1079 images processed...
  ✔ 550/1079 images processed...
  ✔ 600/1079 images processed...
  ✔ 650/1079 images processed...
  ✔ 700/1079 images processed...
  ✔ 750/1079 images processed...
  ✔ 800/1079 images processed...
  ✔ 850/1079 images processed...
  ✔ 900/1079 images processed...
  ✔ 950/1079 images processed...
  ✔ 1000/1079 images processed...
  ✔ 1050/1079 images processed...

✔ PART 1 COMPLETE — 1079 images detected


### Scene Classification

In [8]:
print("\n" + "=" * 55)
print("  PART 2 — SCENE CLASSIFICATION + OCR")
print("=" * 55)

def classify_scene(labels):
    """Classify scene type based on detected objects."""
    label_set = set(labels)
    if "fire" in label_set and "smoke" in label_set:
        return "Fire Scene"
    if "fire" in label_set:
        return "Fire Scene"
    if "smoke" in label_set:
        return "Fire Scene"
    return "General Incident"

print("✔ Scene classification defined")


  PART 2 — SCENE CLASSIFICATION + OCR
✔ Scene classification defined


### OCR — Extract Visible Text

In [9]:
def extract_text_ocr(image_path):
    """
    Use pytesseract OCR to extract visible text from image.
    Captures license plates, street signs, banners.
    """
    try:
        img  = Image.open(image_path)
        text = pytesseract.image_to_string(img, config="--psm 11")
        lines = [ln.strip() for ln in text.split("\n")
                 if len(ln.strip()) >= 3 and any(c.isalnum() for c in ln)]
        return "; ".join(lines[:3]) if lines else "None"
    except Exception:
        return "None"

print("✔ OCR function defined")

✔ OCR function defined


### Process All Detection Results

In [10]:
print("\nProcessing scene classification and OCR...\n")

records = []

if not detection_results:
    print("⚠ No detection results — using demo data")
else:
    for i, det in enumerate(detection_results):
        img_id   = det["Image_ID"]
        img_path = det["image_path"]
        labels   = det["labels"]
        boxes    = det["boxes"]
        avg_conf = det["avg_conf"]

        # Scene classification
        scene_type = classify_scene(labels)

        # Objects detected
        objects_str  = ", ".join(sorted(set(labels))) if labels else "None"

        # Bounding boxes — matching assignment format exactly
        bbox_summary = format_bounding_boxes(boxes)

        # OCR
        text_extracted = extract_text_ocr(img_path)

        records.append({
            "Image_ID":         img_id,
            "Scene_Type":       scene_type,
            "Objects_Detected": objects_str,
            "Bounding_Boxes":   bbox_summary,
            "Text_Extracted":   text_extracted,
            "Confidence":       avg_conf,
        })

        if (i + 1) % 50 == 0:
            print(f"  ✔ {i+1}/{len(detection_results)} processed...")

    print(f"\n✔ {len(records)} records processed")


Processing scene classification and OCR...

  ✔ 50/1079 processed...
  ✔ 100/1079 processed...
  ✔ 150/1079 processed...
  ✔ 200/1079 processed...
  ✔ 250/1079 processed...
  ✔ 300/1079 processed...
  ✔ 350/1079 processed...
  ✔ 400/1079 processed...
  ✔ 450/1079 processed...
  ✔ 500/1079 processed...
  ✔ 550/1079 processed...
  ✔ 600/1079 processed...
  ✔ 650/1079 processed...
  ✔ 700/1079 processed...
  ✔ 750/1079 processed...
  ✔ 800/1079 processed...
  ✔ 850/1079 processed...
  ✔ 900/1079 processed...
  ✔ 950/1079 processed...
  ✔ 1000/1079 processed...
  ✔ 1050/1079 processed...

✔ 1079 records processed


### Demo Data (fallback)

In [11]:
def generate_demo_data():
    return [
        {"Image_ID": "IMG_034", "Scene_Type": "Fire Scene",
         "Objects_Detected": "fire, smoke", "Bounding_Boxes": "2 fire regions, 1 smoke plume",
         "Text_Extracted": "DANGER; NO ENTRY", "Confidence": 0.94},
        {"Image_ID": "IMG_035", "Scene_Type": "Fire Scene",
         "Objects_Detected": "fire", "Bounding_Boxes": "1 fire region",
         "Text_Extracted": "None", "Confidence": 0.88},
        {"Image_ID": "IMG_036", "Scene_Type": "Fire Scene",
         "Objects_Detected": "fire, smoke", "Bounding_Boxes": "3 fire regions, 2 smoke plumes",
         "Text_Extracted": "None", "Confidence": 0.91},
        {"Image_ID": "IMG_037", "Scene_Type": "Fire Scene",
         "Objects_Detected": "smoke", "Bounding_Boxes": "1 smoke plume",
         "Text_Extracted": "None", "Confidence": 0.79},
        {"Image_ID": "IMG_038", "Scene_Type": "Fire Scene",
         "Objects_Detected": "fire, smoke", "Bounding_Boxes": "2 fire regions, 1 smoke plume",
         "Text_Extracted": "EMERGENCY", "Confidence": 0.93},
    ]

if not records:
    records = generate_demo_data()

### Save and Display Output

In [14]:
output_df = pd.DataFrame(records)

# Save full output
output_df.to_csv(OUTPUT_CSV.replace(".csv", "_full.csv"), index=False)

# Assignment output — exact columns
assignment_df = output_df[["Image_ID", "Scene_Type", "Objects_Detected",
                             "Bounding_Boxes", "Confidence"]].copy()
assignment_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✔ image_output.csv saved      — {len(assignment_df)} records")
print(f"✔ image_output_full.csv saved — includes OCR text")

print(f"\n{'='*55}")
print("  STRUCTURED OUTPUT TABLE")
print(f"{'='*55}")
display(assignment_df.head(33))


✔ image_output.csv saved      — 1079 records
✔ image_output_full.csv saved — includes OCR text

  STRUCTURED OUTPUT TABLE


,Image_ID,Scene_Type,Objects_Detected,Bounding_Boxes,Confidence
0,IMG_001,Fire Scene,fire,1 fire region,0.82
1,IMG_002,Fire Scene,fire,1 fire region,0.74
2,IMG_003,Fire Scene,fire,1 fire region,0.93
3,IMG_004,Fire Scene,fire,1 fire region,0.94
4,IMG_005,Fire Scene,fire,1 fire region,0.88
5,IMG_006,Fire Scene,fire,1 fire region,0.95
6,IMG_007,Fire Scene,fire,1 fire region,0.92
7,IMG_008,Fire Scene,fire,1 fire region,0.85
8,IMG_009,Fire Scene,fire,1 fire region,0.88
9,IMG_010,Fire Scene,fire,1 fire region,0.88


### Verify Output Columns

In [13]:
print(f"\n{'='*55}")
print("  OUTPUT COLUMNS CHECK")
print(f"{'='*55}")
expected = ["Image_ID", "Scene_Type", "Objects_Detected",
            "Bounding_Boxes", "Confidence"]
for col in expected:
    status = "✔" if col in assignment_df.columns else "❌"
    print(f"  {status} {col}")

print(f"\n{'='*55}")
print("  SCENE TYPE DISTRIBUTION")
print(f"{'='*55}")
display(assignment_df["Scene_Type"].value_counts().reset_index().rename(
    columns={"index": "Scene_Type", "Scene_Type": "Count"}
))

print(f"\n✔ Download image_output.csv from your Google Drive")
print(f"  Share with your team for the integration step.")


  OUTPUT COLUMNS CHECK
  ✔ Image_ID
  ✔ Scene_Type
  ✔ Objects_Detected
  ✔ Bounding_Boxes
  ✔ Confidence

  SCENE TYPE DISTRIBUTION


,Count,count
0,General Incident,553
1,Fire Scene,526



✔ Download image_output.csv from your Google Drive
  Share with your team for the integration step.
